# House Prices — Corrected Encoding Techniques Submission

**Student:** Claire Panashe Tsuro

## Objective

The goal of this assignment is to prepare the Ames House Prices data for machine learning by **reasoning about the meaning and type of each variable before encoding it**.

The key decisions in this notebook are:

1. Treat `MSSubClass` as a **categorical** variable even though it is stored as an integer.
2. Distinguish **structural NA values** (for example, no basement, no garage, no pool) from genuine missing observations.
3. Use **Ordinal Encoding** for variables whose categories have a meaningful order, especially quality/condition variables.
4. Use **One-Hot Encoding** for the remaining nominal categorical variables, where there is no natural ranking.
5. Fit preprocessing only on the training data and evaluate the complete pipeline on an unseen validation set to avoid data leakage.

This is different from simply converting every categorical column to arbitrary integers.


## 1. Import libraries and load the data

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.max_columns", 100)

train_path = "train.csv"
test_path = "test.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print("Training data shape:", df_train.shape)
print("Test data shape:", df_test.shape)

df_train.head()


## 2. Understand the target and feature data

`SalePrice` is the target variable because it is the value we want the model to predict.

Before encoding, I inspect the data types and missing values. This is important because the correct encoding depends on what the variable actually represents.


In [ ]:
X = df_train.drop(columns="SalePrice").copy()
y = df_train["SalePrice"].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

display(df_train.dtypes.value_counts())


## 3. Inspect missing values before encoding

The missing-value pattern is important in this dataset.

A missing value does **not always mean that information was forgotten**. In the House Prices data, many `NA` values represent the absence of a feature. For example, `PoolQC = NA` means the property has no pool, and `GarageQual = NA` means there is no garage.

Therefore, replacing every categorical missing value with the same generic treatment without considering the meaning would lose information.


In [ ]:
missing = X.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

print("Columns containing missing values:")
display(missing.to_frame("missing_count"))


## 4. Correct the meaning of structural missing values

I separate the missing values into two groups:

### Structural missing values
These mean that the feature does not exist for the house. Examples include:

- `Alley` → no alley access
- `BsmtQual`, `BsmtCond`, `BsmtExposure`, `BsmtFinType1`, `BsmtFinType2` → no basement
- `FireplaceQu` → no fireplace
- `GarageType`, `GarageFinish`, `GarageQual`, `GarageCond` → no garage
- `PoolQC` → no pool
- `Fence` → no fence
- `MiscFeature` → no miscellaneous feature
- `MasVnrType` → no masonry veneer

These are encoded as the explicit category `None`.

For the corresponding numeric structural variables:

- `MasVnrArea` → 0 when there is no masonry veneer
- `GarageYrBlt` → 0 when there is no garage

### Genuine missing values

`LotFrontage` is a genuine numerical missing value, so it is imputed with the training-data median.

`Electrical` has one missing categorical observation, so it is treated as a genuine missing value and filled with the training-data mode.

This distinction preserves the difference between **"this feature does not exist"** and **"this value was not recorded."**


In [ ]:
# Categorical columns where NA means that the feature is absent.
structural_categorical = [
    "Alley", "MasVnrType",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "PoolQC", "Fence", "MiscFeature"
]

for col in structural_categorical:
    X[col] = X[col].fillna("None")

# Structural numerical missing values: feature absent.
X["MasVnrArea"] = X["MasVnrArea"].fillna(0)
X["GarageYrBlt"] = X["GarageYrBlt"].fillna(0)

# Genuine missing values.
X["LotFrontage"] = X["LotFrontage"].fillna(X["LotFrontage"].median())
X["Electrical"] = X["Electrical"].fillna(X["Electrical"].mode()[0])

print("Remaining missing values:", X.isna().sum().sum())


## 5. Fix `MSSubClass` before encoding

`MSSubClass` is stored as an integer, but the values are **codes for dwelling types**, not quantities where arithmetic makes sense.

For example, 20, 30, 40 and 60 identify different dwelling classes. Treating these codes as continuous numerical measurements could incorrectly imply that class 60 is "twice" class 30.

Therefore, I convert `MSSubClass` from integer to string so that it is handled as a categorical feature.

This is an important preprocessing decision made from the meaning of the data, not just from its pandas data type.


In [ ]:
print("MSSubClass before:", X["MSSubClass"].dtype)

X["MSSubClass"] = X["MSSubClass"].astype(str)

print("MSSubClass after:", X["MSSubClass"].dtype)
print("MSSubClass categories:", sorted(X["MSSubClass"].unique()))


## 6. Identify ordinal variables

Ordinal encoding is appropriate when categories have a real order.

For example, house quality follows the order:

`Poor < Fair < Typical/Average < Good < Excellent`

The same idea applies to several basement, garage, heating, kitchen, fireplace, pool, fence and functionality variables.

The category order is explicitly supplied to `OrdinalEncoder` so that the model receives meaningful numerical rankings rather than arbitrary alphabetical codes.


In [ ]:
ordinal_categories = {
    "ExterQual":   ["Po", "Fa", "TA", "Gd", "Ex"],
    "ExterCond":   ["Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtQual":    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtCond":    ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    "BsmtExposure":["None", "No", "Mn", "Av", "Gd"],
    "BsmtFinType1":["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFinType2":["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "HeatingQC":   ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "FireplaceQu": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageFinish":["None", "Unf", "RFn", "Fin"],
    "GarageQual":  ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    "GarageCond":  ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    "PoolQC":      ["None", "Fa", "TA", "Gd", "Ex"],
    "Functional":  ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "Fence":       ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"],
    "LotShape":    ["IR3", "IR2", "IR1", "Reg"],
    "LandSlope":   ["Sev", "Mod", "Gtl"]
}

ordinal_features = list(ordinal_categories.keys())

print("Number of ordinal features:", len(ordinal_features))
print(ordinal_features)


## 7. One-Hot Encoding for the remaining categorical variables

The remaining categorical variables are treated as **nominal** variables because their categories do not have a meaningful ranking.

Examples include:

- `Neighborhood`
- `MSZoning`
- `Exterior1st`
- `Foundation`
- `SaleType`
- `SaleCondition`

For these variables, One-Hot Encoding creates a separate binary feature for each category.

This avoids imposing a false relationship such as:

`Neighborhood A < Neighborhood B < Neighborhood C`

when no such ranking exists.


In [ ]:
all_categorical = X.select_dtypes(include="object").columns.tolist()
nominal_features = [c for c in all_categorical if c not in ordinal_features]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Ordinal categorical features:", len(ordinal_features))
print("Nominal categorical features:", len(nominal_features))
print("Numerical features:", len(numeric_features))

print("\nNominal features:")
print(nominal_features)


## 8. Train/validation split

I split the data before fitting the encoders.

This is important because the encoder must learn category information from the training data only. The validation data must remain unseen during preprocessing and model training.

This gives a more honest estimate of how the model performs on unseen houses.


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))


## 9. Build the encoding pipeline

The preprocessing pipeline contains three branches:

1. **Ordinal features** → imputation + explicitly ordered Ordinal Encoding.
2. **Nominal categorical features** → imputation + One-Hot Encoding.
3. **Numerical features** → median imputation.

The `ColumnTransformer` applies the correct treatment to each group.


In [ ]:
ordinal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[ordinal_categories[col] for col in ordinal_features],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

nominal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("ordinal", ordinal_pipeline, ordinal_features),
        ("nominal", nominal_pipeline, nominal_features),
        ("numeric", numeric_pipeline, numeric_features)
    ]
)

print("Preprocessor created successfully.")


## 10. Validate the encoding itself

Before training the model, I fit the preprocessor on the training data and inspect the transformed shape.

The original dataset has 80 predictor columns. One-Hot Encoding expands the nominal categorical variables into multiple binary columns, while ordinal variables remain as single numerical columns.

This confirms that actual encoding has been applied.


In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_valid_encoded = preprocessor.transform(X_valid)

print("Original training feature count:", X_train.shape[1])
print("Encoded training feature count:", X_train_encoded.shape[1])
print("Encoded validation shape:", X_valid_encoded.shape)

print("Encoded training data contains NaN:", np.isnan(X_train_encoded).sum())


## 11. Train a validated Random Forest model

I now carry the encoding through to a machine learning model.

A Random Forest Regressor is used because `SalePrice` is a continuous numerical target and the relationship between house characteristics and price can be nonlinear.

The model is evaluated only on the held-out validation set.


In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    max_features=0.8
)

model.fit(X_train_encoded, y_train)

y_pred = model.predict(X_valid_encoded)

rmse = mean_squared_error(y_valid, y_pred) ** 0.5
mae = mean_absolute_error(y_valid, y_pred)
r2 = r2_score(y_valid, y_pred)

print(f"Validation RMSE: ${rmse:,.2f}")
print(f"Validation MAE:  ${mae:,.2f}")
print(f"Validation R²:   {r2:.4f}")


## 12. Interpretation of the results

The validation metrics give three views of model performance:

- **RMSE** penalizes large prediction errors more heavily and is useful for comparing house-price predictions.
- **MAE** gives the average absolute prediction error in price units.
- **R²** measures the proportion of variation in `SalePrice` explained by the model.

The most important point for this assignment is that the model was trained **after the categorical variables were actually encoded**, using an encoding method selected according to the meaning of each variable.


## 13. Why I did not use Label Encoding for every categorical feature

Applying Label Encoding to every categorical column would be easy, but it is not always appropriate.

For a nominal variable such as `Neighborhood`, assigning:

`Neighborhood A = 0`, `Neighborhood B = 1`, `Neighborhood C = 2`

would introduce an artificial numerical order.

Instead:

- **Ordinal variables** receive ordered numerical values.
- **Nominal variables** receive One-Hot Encoding.
- **Structural NA values** retain their meaning as "feature absent".
- `MSSubClass` is converted to categorical before encoding.

This makes the preprocessing decisions data-driven rather than simply applying one encoding technique to every column.


## 14. Final encoding strategy

| Variable type | Example | Treatment | Reason |
|---|---|---|---|
| Ordered categorical | `ExterQual`, `KitchenQual`, `BsmtQual` | Ordinal Encoding | Categories have a meaningful rank |
| Nominal categorical | `Neighborhood`, `MSZoning`, `SaleType` | One-Hot Encoding | No natural ranking |
| Structural categorical NA | `PoolQC`, `GarageQual`, `FireplaceQu` | `None` then encode | NA means the feature does not exist |
| Coded categorical | `MSSubClass` | Convert to string, then One-Hot | Integer values are category codes |
| Numerical | `LotArea`, `GrLivArea`, `YearBuilt` | Keep numerical | Already quantitative |
| Genuine numerical missing | `LotFrontage` | Median imputation | Missing observation, not feature absence |
| Structural numerical missing | `MasVnrArea`, `GarageYrBlt` | 0 | Feature is absent |

## Conclusion

The corrected approach does not treat all categorical variables in the same way. I first inspected what the variables mean, distinguished structural absence from genuine missingness, corrected the type of `MSSubClass`, used explicit ordinal mappings for ranked variables, and One-Hot Encoded the remaining categorical variables.

Finally, I validated the complete encoded dataset with a Random Forest regression model on unseen validation data.

**This demonstrates that encoding was actually applied to the House Prices dataset and that the encoding choices were justified by the structure and meaning of the variables.**
